Data Cleaning Pipeline
1. **Load Data**
2. **Data Inspection**
3. **Data Cleaning**
4. **Data Transformation**
5. **Processed Dataset**


## 1. Load Data

Import libraries and load the raw dataset from `../data/raw/automobileEDA_dirty_training.csv`.


In [25]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# path for input and output
raw_data_path = os.path.join('..', 'data', 'raw', 'automobileEDA_dirty_training.csv')
output_data_path = os.path.join('..', 'data', 'processed', 'automobile_processed_v2.csv')

# Load raw dataset
df_raw = pd.read_csv(raw_data_path)
print(f"Shape: {df_raw.shape}")
df_raw.head()


Shape: (205, 30)


,transaction_date,symboling,normalized-losses,make,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,city-L/100km,horsepower-binned,diesel,gas
0,2025-01-01,3,122,alfa-romero,std,two,convertible,rwd,front,88.6,...,9.0,111.0,5000.0,21,27,13495.0,11.190476,Medium,0,1
1,02/01/2025,3,122,alfa-romero,std,two,convertible,rwd,front,88.6,...,9.0,111.0,5000.0,21,27,16500.0,11.190476,Medium,0,1
2,01-03-2025,1,122,ALFA-ROMERO,std,two,hatchback,rwd,front,94.5,...,9.0,154.0,5000.0,19,26,16500.0,12.368421,Medium,0,1
3,04-Jan-2025,2,164,audi,std,four,sedan,fwd,front,99.8,...,10.0,102.0,5500.0,24,30,13950.0,9.791667,Medium,0,1
4,2025-01-05,2,164,audi,std,four,sedan,4wd,front,99.4,...,8.0,115.0,5500.0,18,22,17450.0,13.055556,Medium,0,1


## 2. Data Inspection

Inspect the raw data structure, distributions, missing values, duplicates, and categorical differences.


In [26]:
# Inspect dataset data types
df_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_date   203 non-null    str    
 1   symboling          205 non-null    int64  
 2   normalized-losses  205 non-null    int64  
 3   make               203 non-null    str    
 4   aspiration         205 non-null    str    
 5   num-of-doors       203 non-null    str    
 6   body-style         205 non-null    str    
 7   drive-wheels       205 non-null    str    
 8   engine-location    205 non-null    str    
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    str    
 15  num-of-cylinders   205 non-null    str    
 16  engine-size        205 non-null    in

In [27]:
# Check attributes
df_raw.describe()


,symboling,normalized-losses,wheel-base,length,width,height,curb-weight,engine-size,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,city-L/100km,diesel,gas
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,201.000000,205.000000,202.000000,205.000000,205.000000,205.000000,202.000000,205.000000,205.000000,205.000000
mean,0.839024,122.058537,98.784878,0.837107,0.915095,53.762927,2554.575610,126.980488,3.329020,3.258209,10.207902,103.403526,5119.515800,25.200000,30.721951,13181.668317,9.939602,0.102439,0.897561
std,1.247916,31.748344,6.018632,0.059030,0.028986,2.439080,515.016952,41.398466,0.266929,0.316354,4.064078,37.453870,476.073141,6.446917,6.836734,7924.054951,2.537799,0.303967,0.303967
min,-2.000000,65.000000,86.600000,0.678039,0.837500,47.800000,1488.000000,61.000000,2.540000,2.070000,7.000000,48.000000,4150.000000,13.000000,16.000000,5118.000000,4.795918,0.000000,0.000000
25%,0.000000,101.000000,94.500000,0.801538,0.890278,52.000000,2169.000000,98.000000,3.150000,3.110000,8.600000,70.000000,4800.000000,19.000000,25.000000,7778.250000,7.833333,0.000000,1.000000
50%,1.000000,122.000000,97.000000,0.832292,0.909722,54.100000,2420.000000,120.000000,3.310000,3.290000,9.000000,95.000000,5200.000000,24.000000,30.000000,10320.000000,9.791667,0.000000,1.000000
75%,2.000000,137.000000,102.400000,0.881788,0.923611,55.500000,2926.000000,141.000000,3.580000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.000000,16482.500000,12.368421,0.000000,1.000000
max,3.000000,256.000000,120.900000,1.000000,1.000000,59.800000,4066.000000,326.000000,3.940000,4.170000,23.000000,262.000000,6600.000000,49.000000,54.000000,45400.000000,18.076923,1.000000,1.000000


In [28]:
# Check missing values count per column
missing_vals = df_raw.isnull().sum()
missing_vals = missing_vals[missing_vals > 0]
print("Missing values per column:")
print(missing_vals)


Missing values per column:
transaction_date     2
make                 2
num-of-doors         2
stroke               4
horsepower           3
price                3
horsepower-binned    1
dtype: int64


In [29]:
# Check duplicate rows
duplicates_count = df_raw.duplicated().sum()
print(f"Duplicate rows: {duplicates_count}")
df_raw[df_raw.duplicated(keep=False)].head(6)


Duplicate rows: 4


,transaction_date,symboling,normalized-losses,make,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,city-L/100km,horsepower-binned,diesel,gas
5,06/01/2025,2,122,audi,std,two,sedan,fwd,front,99.8,...,8.5,110.0,5500.0,19,25,15250.0,12.368421,Medium,0,1
25,26/01/2025,1,148,dodge,std,four,sedan,fwd,front,93.7,...,9.4,68.0,5500.0,31,38,7609.0,7.580645,Low,0,1
60,2025-03-02,0,122,mazda,std,four,sedan,fwd,front,98.8,...,22.7,64.0,4650.0,36,42,10795.0,6.527778,Low,1,0
100,2025-04-11,0,108,NaN,std,four,sedan,fwd,front,100.4,...,9.0,152.0,5200.0,19,25,13499.0,12.368421,Medium,0,1
201,06/01/2025,2,122,audi,std,two,sedan,fwd,front,99.8,...,8.5,110.0,5500.0,19,25,15250.0,12.368421,Medium,0,1
202,26/01/2025,1,148,dodge,std,four,sedan,fwd,front,93.7,...,9.4,68.0,5500.0,31,38,7609.0,7.580645,Low,0,1


In [30]:
# Check category variables for inconsistent format
cat_cols = ['make', 'body-style', 'drive-wheels', 'fuel-system', 'num-of-doors', 'num-of-cylinders', 'horsepower-binned']
for col in cat_cols:
    print(f"Column: {col}")
    print(df_raw[col].unique())
    print()


Column: make
<StringArray>
[  'alfa-romero',   'ALFA-ROMERO',          'audi',           'bmw',
     'chevrolet',         'dodge',       'dodge  ',         'honda',
          'Audi',         'isuzu',        'jaguar',         'mazda',
 'mercedes-benz',     'mercury  ',    'mitsubishi',           'BMW',
        'nissan',             nan,        'peugot',        'toyota',
      'plymouth',       'porsche',     'porsche  ',       'renault',
          'saab',        'subaru',    'volkswagen',         'volvo']
Length: 28, dtype: str

Column: body-style
<StringArray>
['convertible', 'hatchback', 'sedan', 'wagon', 'SEDAN', 'Sedan', 'hardtop']
Length: 7, dtype: str

Column: drive-wheels
<StringArray>
['rwd', 'fwd', '4wd', 'AWD', 'RWD']
Length: 5, dtype: str

Column: fuel-system
<StringArray>
['mpfi', '2bbl', 'MPFI', 'mfi', '1bbl', 'spfi', '4bbl', 'idi', 'Mpfi', 'spdi']
Length: 10, dtype: str

Column: num-of-doors
<StringArray>
['two', 'four', nan]
Length: 3, dtype: str

Column: num-of-cylinders

## 3. Data Cleaning

We do :
- Duplicate data
- Normalize data
- Deal with missing values (Nan)
  - `price`,`horsepower`,`stroke` = use column medians.
  - `num-of-doors`,`make`,`horsepower-binned` = column modes.


In [31]:
df_clean = df_raw.copy()

# 1. Remove duplicate
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df_clean.shape}")


Shape after removing duplicates: (201, 30)


In [32]:
# 2. String normalization
string_cols = ['make', 'body-style', 'drive-wheels', 'fuel-system', 'engine-type', 'aspiration', 'engine-location', 'num-of-doors']
for col in string_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
    df_clean.loc[df_clean[col] == 'nan', col] = np.nan

print("Normalized body-style:", df_clean['body-style'].unique())
print("Normalized drive-wheels:", df_clean['drive-wheels'].unique())
print("Normalized fuel-system:", df_clean['fuel-system'].unique())


Normalized body-style: <StringArray>
['convertible', 'hatchback', 'sedan', 'wagon', 'hardtop']
Length: 5, dtype: str
Normalized drive-wheels: <StringArray>
['rwd', 'fwd', '4wd', 'awd']
Length: 4, dtype: str
Normalized fuel-system: <StringArray>
['mpfi', '2bbl', 'mfi', '1bbl', 'spfi', '4bbl', 'idi', 'spdi']
Length: 8, dtype: str


In [33]:
# 3. Missing Value
num_impute_cols = ['price', 'stroke', 'horsepower']
for col in num_impute_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"{col} missing values with median: {median_val}")

cat_impute_cols = ['num-of-doors', 'make', 'horsepower-binned']
for col in cat_impute_cols:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"{col} missing values with mode: {mode_val}")

remaining_nulls = df_clean.drop(columns=['transaction_date']).isnull().sum().sum()
print(f"\nRemain missing values: {remaining_nulls}")


price missing values with median: 10270.0
stroke missing values with median: 3.29
horsepower missing values with median: 95.0
num-of-doors missing values with mode: four
make missing values with mode: toyota
horsepower-binned missing values with mode: Low

Remain missing values: 0


## 4. Data Transformation

- Encoding:
  - `num-of-doors`: `two`: 0, `four`: 1.
  - `num-of-cylinders`: `two`: 2, `three`: 3, `four`: 4, `five`: 5, `six`: 6, `eight`: 8, `twelve`: 12.
  - `horsepower-binned`: `Low`: 0, `Medium`: 1, `High`: 2) as `horsepower_ordinal`.
- Frequency Encoding:
  - `make_freq`: mapped from manufacturer relative frequencies.
- Min-Max Scaling:
  - Continuous features scaled to the range `[0, 1]`.
- One hot encoding:
  - Categorical nominal attributes (`body-style`, `drive-wheels`, `aspiration`, `engine-type`, `engine-location`, `fuel-system`).
- Feature Selection / Cleanup:
  - Exclude `transaction_date` and raw categorical columns because unrelevant.


In [34]:
df_transformed = df_clean.copy()

# 1. Map num-of-doors
door_mapping = {'two': 0, 'four': 1}
df_transformed['num-of-doors'] = df_transformed['num-of-doors'].map(door_mapping)

# 2. Map num-of-cylinders
cyl_mapping = {
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5,
    'six': 6,
    'eight': 8,
    'twelve': 12
}
df_transformed['num-of-cylinders'] = df_transformed['num-of-cylinders'].map(cyl_mapping)

# 3. Create ordinal feature for horsepower-binned
hp_order_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df_transformed['horsepower_ordinal'] = df_transformed['horsepower-binned'].map(hp_order_mapping)

# 4. Frequency encoding for 'make'
make_frequencies = df_transformed['make'].value_counts(normalize=True)
df_transformed['make_freq'] = df_transformed['make'].map(make_frequencies)

# Display sample of engineered features
df_transformed[['num-of-doors', 'num-of-cylinders', 'horsepower_ordinal', 'make_freq']].head()


,num-of-doors,num-of-cylinders,horsepower_ordinal,make_freq
0,0,4,1,0.014925
1,0,4,1,0.014925
2,0,6,1,0.014925
3,1,4,1,0.034826
4,1,5,1,0.034826


In [35]:
# 5. Min-Max Scaling
scale_columns = [
    'symboling',
    'curb-weight',
    'num-of-cylinders',
    'engine-size',
    'horsepower',
    'peak-rpm',
    'city-mpg',
    'highway-mpg',
    'price'
]

scaler = MinMaxScaler()
df_transformed[scale_columns] = scaler.fit_transform(df_transformed[scale_columns])
df_transformed[scale_columns].describe()


,symboling,curb-weight,num-of-cylinders,engine-size,horsepower,peak-rpm,city-mpg,highway-mpg,price
count,201.000000,201.000000,201.000000,201.000000,201.000000,201.000000,201.000000,201.000000,201.000000
mean,0.568159,0.414145,0.236318,0.248587,0.258765,0.394965,0.338308,0.386489,0.199790
std,0.250960,0.200658,0.105945,0.156781,0.173893,0.195148,0.178423,0.179346,0.197083
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.400000,0.264158,0.200000,0.139623,0.102804,0.265306,0.166667,0.236842,0.066283
50%,0.600000,0.359193,0.200000,0.222642,0.219626,0.398110,0.305556,0.368421,0.127898
75%,0.800000,0.557797,0.200000,0.301887,0.317757,0.551020,0.472222,0.473684,0.282558
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [36]:
# 6. One hot encoding for nominal categorical columns
ohe_columns = ['body-style', 'drive-wheels', 'aspiration', 'engine-type', 'engine-location', 'fuel-system']
df_dummies = pd.get_dummies(df_transformed[ohe_columns], dtype=int)

# 7. Drop categorical and metadata columns
drop_columns = ohe_columns + ['transaction_date', 'make', 'horsepower-binned']
df_features = df_transformed.drop(columns=drop_columns)
df_processed = pd.concat([df_features, df_dummies], axis=1)

print(f"Transformed dataset shape: {df_processed.shape}")
df_processed.head()


Transformed dataset shape: (201, 50)


,symboling,normalized-losses,num-of-doors,wheel-base,length,width,height,curb-weight,num-of-cylinders,engine-size,...,engine-location_front,engine-location_rear,fuel-system_1bbl,fuel-system_2bbl,fuel-system_4bbl,fuel-system_idi,fuel-system_mfi,fuel-system_mpfi,fuel-system_spdi,fuel-system_spfi
0,1.0,122,0,88.6,0.811148,0.890278,48.8,0.411171,0.2,0.260377,...,1,0,0,0,0,0,0,1,0,0
1,1.0,122,0,88.6,0.811148,0.890278,48.8,0.411171,0.2,0.260377,...,1,0,0,0,0,0,0,1,0,0
2,0.6,122,0,94.5,0.822681,0.909722,52.4,0.517843,0.4,0.343396,...,1,0,0,0,0,0,0,1,0,0
3,0.8,164,1,99.8,0.848630,0.919444,54.3,0.329325,0.2,0.181132,...,1,0,0,0,0,0,0,1,0,0
4,0.8,164,1,99.4,0.848630,0.922222,54.3,0.518231,0.3,0.283019,...,1,0,0,0,0,0,0,1,0,0


## 5. Processed Dataset

Verify the final dataset before saving to `../data/processed/automobile_processed_v2.csv`.


In [37]:
# Verify no null values
total_nulls = df_processed.isnull().sum().sum()
print(f"Missing values : {total_nulls}")

# Verify final data shape
print(f"Final Dataset Shape: {df_processed.shape} (Rows: {df_processed.shape[0]}, Columns: {df_processed.shape[1]})")


Missing values : 0
Final Dataset Shape: (201, 50) (Rows: 201, Columns: 50)


In [38]:
# Export processed data to CSV
os.makedirs(os.path.dirname(output_data_path), exist_ok=True)
df_processed.to_csv(output_data_path, index=False)
print(f"Successfully exported processed dataset to: {output_data_path}")


Successfully exported processed dataset to: ..\data\processed\automobile_processed_v2.csv


In [39]:
# Compare to reference automobile_processed.csv
ref_processed_path = os.path.join('..', 'data', 'processed', 'automobile_processed.csv')
if os.path.exists(ref_processed_path):
    df_ref = pd.read_csv(ref_processed_path)
    # Align column ordering
    df_processed_aligned = df_processed[df_ref.columns]
    
    print(f"Reference shape: {df_ref.shape}")
    print(f"Pipeline shape: {df_processed_aligned.shape}")
    
    cols_match = list(df_processed_aligned.columns) == list(df_ref.columns)
    print(f"Shape reference : {cols_match}")



Reference shape: (201, 50)
Pipeline shape: (201, 50)
Shape reference : True
